<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## Week 6 · Session 2 · Cleaning, Grouping, and Merging Data with pandas

        **Course level:** absolute beginner  
        **Prior learning:** DataFrame creation, inspection, selection, and filtering from Session 6.1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Real datasets contain missing values, inconsistent text, duplicates, and separated tables. This session develops a documented cleaning workflow before introducing grouped summaries and joins.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - profile missing values and duplicates
- choose and justify a missing-data treatment
- clean text and convert data types
- summarise groups with `groupby` and `agg`
- merge tables and check whether rows matched as expected


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Recall | 10 min | Retrieve earlier ideas with a small runnable example |
| Learn | 30 min | Add new concepts in short, explained steps |
| Guided lab | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation first, predict the output, then run the code with <code>Shift + Enter</code>. Only edit cells marked <strong>Student task</strong>. Ask what every line contributes.</div>


### Recall: create and inspect a DataFrame

The session is self-contained: it imports pandas and recreates the data it needs.


In [ ]:
import pandas as pd

raw_orders = pd.DataFrame({
    "order_id": [1, 2, 3, 3, 4],
    "region": [" North", "south", "NORTH", "NORTH", None],
    "quantity": [2, 1, None, None, 4],
    "unit_price": [10.0, 15.5, 8.0, 8.0, 12.0],
    "order_date": ["2026-01-03", "2026-01-04", "bad date", "bad date", "2026-01-06"],
})

print(raw_orders)
print("Shape:", raw_orders.shape)


## 1. Profile quality before changing data

Cleaning is a decision process, not a button. First measure the problems, preserve the raw input, and record why each treatment is appropriate.


### Count missing values and duplicated rows

`.isna()` marks missing cells. Summing Boolean values counts `True`. `.duplicated()` detects repeated rows.


In [ ]:
missing_by_column = raw_orders.isna().sum()
duplicate_count = raw_orders.duplicated().sum()

print("Missing values:", missing_by_column)
print("Duplicate rows:", duplicate_count)


### Inspect categories before standardising them

`value_counts(dropna=False)` includes missing categories and reveals spelling, case, and whitespace inconsistencies.


In [ ]:
print(raw_orders["region"].value_counts(dropna=False))


## 2. Treat missing values deliberately

Common choices include:

- leave missing when absence is meaningful;
- fill using a justified value or statistic;
- drop rows only when the cost is understood;
- seek a corrected source when the field is essential.

Never replace all missing values with zero automatically: zero is a real measurement.


### Fill a numeric value and a category

This teaching example uses the median quantity and an explicit `Unknown` region. A real project must document why those choices are defensible.


In [ ]:
orders = raw_orders.copy()
median_quantity = orders["quantity"].median()

orders["quantity"] = orders["quantity"].fillna(median_quantity)
orders["region"] = orders["region"].fillna("Unknown")

print("Median used:", median_quantity)
print(orders.isna().sum())


### Drop a row only for an essential field

Here we demonstrate the syntax on a separate miniature table so the main workflow remains intact.


In [ ]:
contacts = pd.DataFrame({
    "contact_id": [1, 2, 3],
    "email": ["a@example.org", None, "c@example.org"],
})
contacts_with_email = contacts.dropna(subset=["email"])

print(contacts_with_email)


## 3. Standardise text and remove duplicates

Text can differ because of leading spaces, capitalisation, or inconsistent labels. Apply small, inspectable operations in a logical order.


### Clean a category column

Convert to pandas' string type, remove surrounding whitespace, and use title case. The explicit `Unknown` label remains readable.


In [ ]:
orders["region"] = (
    orders["region"]
    .astype("string")
    .str.strip()
    .str.title()
)

print(orders["region"].value_counts(dropna=False))


### Remove exact duplicate rows

Count before and after so the transformation is auditable. An identical order identifier does not always mean a duplicate, but these rows are exact copies.


In [ ]:
rows_before = len(orders)
orders = orders.drop_duplicates().copy()
rows_after = len(orders)

print("Rows removed:", rows_before - rows_after)
print(orders)


## 4. Convert types safely

Dates and numeric values often arrive as text. `errors="coerce"` converts invalid entries to missing values, which makes the problem measurable rather than silently accepting bad text.


### Parse dates and inspect failures

The invalid date becomes `NaT`, pandas' missing timestamp marker.


In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
invalid_date_count = orders["order_date"].isna().sum()
orders["quantity"] = orders["quantity"].astype("int64")

print(orders.dtypes)
print("Invalid dates:", invalid_date_count)


## 5. Calculate and summarise groups

The **split–apply–combine** idea underlies `groupby`:

1. split rows into groups;
2. apply a calculation within each group;
3. combine the results into a new table.


### Add revenue and calculate group totals

The report contains one row per region because `as_index=False` keeps the group label as a normal column.


In [ ]:
orders["revenue"] = orders["quantity"] * orders["unit_price"]

revenue_by_region = (
    orders.groupby("region", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)

print(revenue_by_region)


### Create several summaries with `.agg()`

Named aggregations produce clear output columns instead of a multi-level header.


In [ ]:
region_summary = (
    orders.groupby("region", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        mean_price=("unit_price", "mean"),
        total_revenue=("revenue", "sum"),
    )
)

print(region_summary)


## 6. Merge related tables

Tables are often separated to avoid repeating descriptive information. A **key** links them. Before merging, ask whether the key should be unique and whether unmatched rows are acceptable.


### Merge orders with customer details

`how="left"` preserves every order. `validate="many_to_one"` checks that many orders may refer to one unique customer record.


In [ ]:
order_lines = pd.DataFrame({
    "order_id": [201, 202, 203, 204],
    "customer_id": [10, 11, 10, 99],
    "amount": [35.0, 42.5, 18.0, 27.0],
})
customers = pd.DataFrame({
    "customer_id": [10, 11, 12],
    "customer_name": ["Asha", "Bilal", "Clara"],
    "segment": ["Retail", "Business", "Retail"],
})

combined = order_lines.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(combined)
print("Match check:", combined["_merge"].value_counts())


## Guided lab: a reproducible cleaning pipeline

A professional workflow preserves raw data, creates a cleaned copy, verifies assumptions after each transformation, and produces a result tied to a clear question.


### Clean and summarise a survey

This compact pipeline standardises groups, converts scores, removes unusable rows, and reports the mean by group.


In [ ]:
survey_raw = pd.DataFrame({
    "participant": [" P01", "P02", "P03", "P04"],
    "group": ["control", "Treatment", " CONTROL ", "treatment"],
    "score": ["72", "88", "not recorded", "91"],
})

survey_clean = survey_raw.copy()
survey_clean["participant"] = survey_clean["participant"].str.strip()
survey_clean["group"] = survey_clean["group"].str.strip().str.title()
survey_clean["score"] = pd.to_numeric(survey_clean["score"], errors="coerce")
survey_clean = survey_clean.dropna(subset=["score"])

survey_summary = survey_clean.groupby("group", as_index=False).agg(
    participants=("participant", "count"),
    mean_score=("score", "mean"),
)

print(survey_clean)
print("Summary:", survey_summary)


## Student coding lab

For every task, state what you changed and check the shape or counts after changing it.


        ### Student task 1: Create a quality report

        <div class="task"><strong>Your job:</strong> For the supplied DataFrame, print missing counts by column, the exact duplicate count, the shape, and all unique department values including missing.</div>

        **Check your work**

        - Missing values are counted per column.
- Exact duplicates are counted.
- The shape and raw categories are visible.


In [ ]:
# STUDENT TASK 1
employees_raw = pd.DataFrame({
    "employee_id": [1, 2, 2, 3],
    "department": ["Sales", " IT ", " IT ", None],
    "hours": [35, 40, 40, None],
})

print(employees_raw)
# Add the four requested quality checks.


        ### Student task 2: Clean missing and repeated data

        <div class="task"><strong>Your job:</strong> Copy `employees_raw`, fill missing department with `Unknown`, fill missing hours with the median, remove exact duplicates, and print before/after row counts.</div>

        **Check your work**

        - The raw DataFrame is preserved.
- No missing values remain in the two specified columns.
- Only exact duplicate rows are removed.


In [ ]:
# STUDENT TASK 2
employees = employees_raw.copy()
rows_before_cleaning = len(employees)

# Add fill and duplicate-removal operations.

rows_after_cleaning = len(employees)
print(employees)
print("Rows before:", rows_before_cleaning)
print("Rows after:", rows_after_cleaning)


        ### Student task 3: Standardise text and numeric types

        <div class="task"><strong>Your job:</strong> Strip spaces and title-case department names. Convert hours to integers, then print value counts and dtypes.</div>

        **Check your work**

        - Whitespace and capitalisation are consistent.
- Hours use an integer dtype.
- The result is inspected after conversion.


In [ ]:
# STUDENT TASK 3
employees_standardised = employees.copy()

# Add text cleaning and type conversion.

print(employees_standardised)
print(employees_standardised.dtypes)


        ### Student task 4: Build a grouped summary

        <div class="task"><strong>Your job:</strong> Using the supplied sales table, produce one row per category with transaction count, total units, and total revenue. Sort from highest to lowest revenue.</div>

        **Check your work**

        - There is exactly one row per category.
- All three requested summaries have clear names.
- The largest revenue is first.


In [ ]:
# STUDENT TASK 4
sales = pd.DataFrame({
    "category": ["A", "B", "A", "B", "A"],
    "units": [2, 5, 4, 3, 1],
    "revenue": [20.0, 45.0, 36.0, 30.0, 12.0],
})

category_summary = sales.copy()  # Replace with groupby, agg, and sort.
print(category_summary)


        ### Student task 5: Merge and audit two tables

        <div class="task"><strong>Your job:</strong> Left-merge visits with clinics on `clinic_id`. Validate a many-to-one relationship, request an indicator column, and print unmatched visits.</div>

        **Check your work**

        - All four visits remain.
- The merge relationship is validated.
- The unmatched clinic key is easy to identify.



**Optional extension:** Also identify clinics that received no visits.


In [ ]:
# STUDENT TASK 5
visits = pd.DataFrame({
    "visit_id": [501, 502, 503, 504],
    "clinic_id": [1, 2, 1, 9],
})
clinics = pd.DataFrame({
    "clinic_id": [1, 2, 3],
    "clinic_name": ["Central", "East", "West"],
})

visit_report = visits.copy()  # Replace with `.merge(...)`.
print(visit_report)
# Print only unmatched rows after adding the indicator.


        ## Knowledge check

        Answer these without running new code first.

        1. Why should raw data be preserved before cleaning?
2. Why is filling every missing value with zero risky?
3. What does `errors="coerce"` do during conversion?
4. What are the three conceptual stages of `groupby`?
5. What can `validate` and `indicator` reveal after a merge?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Review keys, rows, columns, and joins. Week 7 applies these ideas to a relational SQL database through SQLAlchemy.
